<a href="https://colab.research.google.com/github/asmdhabibullah/FinDRL/blob/main/Thesis_Stock_Market_Price_Forecasting_using_Deep_Reinforcement_learning_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print(os.name)

In [ ]:
!pip install pandas numpy pandas_ta yfinance tensorflow scikit-optimize

In [ ]:
import os
import pickle
import inspect
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
from skopt import gp_minimize
from itertools import product
from google.colab import drive
from skopt.space import Integer
from skopt.utils import use_named_args
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model

In [ ]:
drive.mount('/content/drive')

In [ ]:
org_path="/content/drive/MyDrive/StockStrategies/data"

data_path=f"{org_path}/xdu"

if not os.path.exists(data_path):
  os.mkdir(data_path)

dp = f"{data_path}/thesis.csv"

# Load or Download Data
if os.path.exists(dp):
    data = pd.read_csv(dp)
else:
    data = yf.download("^GSPC", interval="1d", period="max", multi_level_index=False)

# Preprocess Data
if 'date' not in data.columns:
    data.reset_index(inplace=True)  # Reset index to bring 'date' as a column

# Convert column names to lowercase
data.columns = data.columns.str.lower()

# Ensure 'date' is the index
data["date"] = pd.to_datetime(data["date"])  # Convert to datetime format
data.set_index("date", inplace=True)

# Save if downloaded
if not os.path.exists(dp):
    data.to_csv(dp)

# Drop 'date' column before passing to the optimizer
numeric_data = data.drop(columns=["date"], errors="ignore")

In [ ]:
# Define Indicator Configurations for Multi-Indicator Optimization
indicator_config = {
    "cci": {"length": [1, 100]},
    "cmo": { "length": [1, 100]},
    "macd": {
        "fast": [1, 100],
        "slow": [1, 100],
        "signal": [1, 100]
    },
    "rsi": {"length": [1, 100] },
    "stoch": {
        "k": [1, 100],
        "d": [1, 100],
        "smooth_k": [1, 100]
    },
    "adosc": {
      "fast": [1, 100],
      "slow": [1, 100]
    },
    "cmf": { "length": [1, 100]},
    "kvo": {
      "fast": [1, 100],
      "slow": [1, 100]
    },
    "mfi": {"length": [1, 100]},
    "obv": {"length": [1, 100]},
    "ao": {
      "fast": [1, 100],
      "slow": [1, 100]
    },
    "apo": {
      "fast": [1, 100],
      "slow": [1, 100],
      "signal": [1, 100]
    },
    "bias": {"length": [1, 100]},
    "brar": {"length": [1, 100]},
    "cfo": { "length": [1, 100]},
    "cg": { "length": [1, 100]},
    "cti": {"length": [1, 100]},
    "er": {"length": [1, 100]},
    "eri": {"length": [1, 100]},
    "fisher": {"length": [1, 100]},
    "inertia": {"length": [1, 100]},
    "coppock": { "length": [1, 100]},
    "kdj": {
      "k": [1, 100],
      "d": [1, 100],
      "smooth_k": [1, 100]
    },
    "kst": {
      "roc1": [1, 100],
      "roc2": [1, 100],
      "roc3": [1, 100],
      "roc4": [1, 100]
    },
    "mom": {"length": [1, 100]},
    "pgo": {"length": [1, 100]},
    "ppo": {
      "fast": [1, 100],
      "slow": [1, 100],
      "signal": [1, 100]
    },
    "roc": { "length": [1, 100]},
    "trix": { "length": [1, 100]},
    "uo": {
      "fast": [1, 100],
      "medium": [1, 100],
      "slow": [1, 100]
    },
    "willr": { "length": [1, 100]},
    "aobv": { "length": [1, 100]},
    "efi": { "length": [1, 100]},
    "eom": { "length": [1, 100]},
    "nvi": {"length": [1, 100]},
    "pvi": {"length": [1, 100]},
    "pvt": {"length": [1, 100]},
}

In [ ]:
class TradingStrategyOptimizer:
    """
    Fully integrated trading strategy optimizer using Deep Learning and Bayesian Optimization.
    Handles training, optimization, saving/loading, and dynamic backtesting for multiple indicators.
    """
    def __init__(self, parameter_ranges, df, save_dir="saved_models"):
        """
        :param parameter_ranges: Dictionary with indicator names as keys and their parameter ranges as values.
        :param df: DataFrame containing OHLCV data.
        :param save_dir: Directory where models and objects will be saved.
        """
        self.df = df  # Store OHLCV data
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)  # Ensure directory exists
        self.parameter_ranges = parameter_ranges
        self.num_params = sum(len(params) for params in parameter_ranges.values())  # Total number of parameters
        self.input_dim = df.shape[1]  # OHLCV features count
        self.scaler = MinMaxScaler()
        self.model = self._build_model()
        self._prepare_data()

    def _build_model(self):
        """
        Build and compile the Deep Learning model.
        """
        model = models.Sequential([
            layers.Input(shape=(self.input_dim,)),
            layers.Dense(128, activation='relu'),
            layers.Dense(64, activation='relu'),
            layers.Dense(64, activation='relu'),
            layers.Dense(self.num_params, activation='sigmoid')  # Normalized output
        ])
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        return model

    def _prepare_data(self):
        """
        Normalize OHLCV data and split into training/testing sets.
        """
        self.data_size = len(self.df)
        X = self.scaler.fit_transform(self.df)  # Normalize OHLCV data
        y = np.random.uniform(0, 1, (self.data_size, self.num_params))  # Simulated targets

        # Train-test split
        split = int(0.8 * self.data_size)
        self.X_train, self.X_test = X[:split], X[split:]
        self.y_train, self.y_test = y[:split], y[split:]

        print(f"✅ Data Prepared: X_train: {self.X_train.shape}, y_train: {self.y_train.shape}")

    def train(self, epochs=10, batch_size=16):
        """
        Train the deep learning model.
        """
        self.model.fit(self.X_train, self.y_train, epochs=epochs, batch_size=batch_size, verbose=1)

    def optimize_all_indicators(self, n_calls=20):
        """
        Optimize all indicators dynamically and return the top 5 best parameters for each.
        """
        optimized_results = {}
        for indicator in self.parameter_ranges.keys():
            optimized_results[indicator] = self.optimize_parameters(indicator, self.X_test, n_calls)
        return optimized_results

    def optimize_parameters(self, indicator_name, X, n_calls=20):
        """
        Use Bayesian Optimization to refine parameters for a specific indicator.
        """
        if indicator_name not in self.parameter_ranges:
            raise ValueError(f"Indicator '{indicator_name}' not found.")

        space = [Integer(*param_range, name=f"{indicator_name}_{param}")
                 for param, param_range in self.parameter_ranges[indicator_name].items()]

        @use_named_args(space)
        def objective(**params):
            return -self._backtest_function(params)  # Minimize negative performance metrics

        res = gp_minimize(objective, space, n_calls=n_calls, random_state=42)

        best_params = [dict(zip([f"{indicator_name}_{param}" for param in self.parameter_ranges[indicator_name]], res.x_iters[i])) for i in range(5)]
        return best_params

    def _backtest_function(self, params):
        """
        Runs a fully dynamic backtest using the available indicators.
        Automatically generates buy/sell signals based on indicator conditions.
        """
        df = self.df.copy()
        buy_signals = pd.Series(False, index=df.index)
        sell_signals = pd.Series(False, index=df.index)

        # Copy OHLCV columns with `_` suffix (Only Once)
        required_columns = ["open", "high", "low", "close", "volume"]
        for col in required_columns:
            df[f"{col}_"] = df[col]  # Create duplicate `_` columns

        for indicator, param_set in self.parameter_ranges.items():
            indicator_func = getattr(ta, indicator.lower(), None)
            if indicator_func:
                try:
                    # Get required arguments for the indicator function
                    indicator_args = inspect.signature(indicator_func).parameters.keys()

                    # Ensure required OHLCV columns exist
                    indicator_inputs = {arg: df[arg] for arg in indicator_args if arg in df.columns}

                    # Add indicator-specific parameters
                    indicator_inputs.update({key.split("_")[-1]: int(params[key]) for key in params if key.startswith(indicator)})

                    # Debugging: Print indicator inputs
                    print(f"Indicator: {indicator}, Inputs: {indicator_inputs.keys()}")

                    # Apply the indicator function
                    result = indicator_func(**indicator_inputs)

                    # Fix dtype issue by enforcing float64 conversion
                    if isinstance(result, pd.DataFrame):
                        for col in result.columns:
                            df[f"{indicator}_{col}"] = result[col].astype("float64", errors="ignore")  # Convert to float64
                    else:
                        df[f"{indicator}"] = pd.to_numeric(result, errors="coerce").astype("float64")  # Force float64

                except Exception as e:
                    print(f"⚠️ Error applying {indicator}: {e}")

        for indicator in self.parameter_ranges:
            if f"{indicator}" in df.columns:
                buy_signals |= df[f"{indicator}"] < df[f"{indicator}"].rolling(5).mean()
                sell_signals |= df[f"{indicator}"] > df[f"{indicator}"].rolling(5).mean()

        df["returns"] = df["close"].pct_change()
        df["strategy_returns"] = df["returns"] * (buy_signals.shift(1) - sell_signals.shift(1))

        sharpe_ratio = df["strategy_returns"].mean() / df["strategy_returns"].std()
        return sharpe_ratio

    def train_and_optimize(self, epochs=5, batch_size=16, n_calls=20):
        """
        Train the model and optimize all indicators.
        """
        print("\n🔹 Training the model...\n")
        self.train(epochs, batch_size)
        print("\n✅ Model training complete!")

        print("\n🔹 Optimizing all indicators...\n")
        optimized_params = self.optimize_all_indicators(n_calls)
        print("\n✅ Optimization complete!\n")
        return optimized_params

    def save_model(self, filename="models.h5"):
        """
        Save the trained model.
        """
        model_path = os.path.join(self.save_dir, filename)
        self.model.save(model_path)
        print(f"✅ Model saved at: {model_path}")

    def save_optimizer(self, filename="optimizer.pkl"):
        """
        Save the entire optimizer object.
        """
        optimizer_path = os.path.join(self.save_dir, filename)
        with open(optimizer_path, "wb") as f:
            pickle.dump(self, f)
        print(f"✅ Optimizer saved at: {optimizer_path}")

    def load_model(self, filename="model.h5"):
        """
        Load a trained model.
        """
        model_path = os.path.join(self.save_dir, filename)
        if os.path.exists(model_path):
            self.model = load_model(model_path)
            print(f"✅ Model loaded from: {model_path}")
        else:
            print(f"❌ Model file not found: {model_path}")

    @staticmethod
    def load_optimizer(filename="trading_optimizer.pkl", save_dir="models"):
        """
        Load the full optimizer object.
        """
        optimizer_path = os.path.join(save_dir, filename)
        if os.path.exists(optimizer_path):
            with open(optimizer_path, "rb") as f:
                optimizer = pickle.load(f)
            print(f"✅ Optimizer loaded from: {optimizer_path}")
            return optimizer
        else:
            print(f"❌ Optimizer file not found: {optimizer_path}")
            return None

In [ ]:
# Load Real OHLCV Data & Initialize Optimizer
multi_optimizer = TradingStrategyOptimizer(indicator_config, numeric_data)

# Train & Optimize
optimized_params = multi_optimizer.train_and_optimize(epochs=5, batch_size=16, n_calls=10)

In [ ]:
# Save Model & Optimizer
multi_optimizer.save_model(f"{data_path}/models.h5")
multi_optimizer.save_optimizer(f"{data_path}/optimizer.pkl")